## Label Generation
Map morphological types to binary labels: 1 = barred, 0 = unbarred.

In [ ]:
import pandas as pd, re

def is_barred(morph_type):
    if pd.isna(morph_type) or morph_type == '':
        return None
    m = str(morph_type).strip()
    if m.startswith('SB') or m.startswith('SAB'):
        return 1
    if m.startswith('SA') or m.startswith('S0') or m.startswith('E'):
        return 0
    if m.startswith('S'):
        return 0
    if m.startswith('(R'):
        hit = re.search(r'\(R[^\)]*\)\s*(.+)', m)
        if hit:
            core = hit.group(1)
            if core.startswith('SB') or core.startswith('SAB'):
                return 1
            if core.startswith('SA'):
                return 0
    return None


In [ ]:
def process_file(inp, out):
    df = pd.read_csv(inp)
    df['Label'] = df['Morphological Type'].apply(is_barred)
    result = df[['Target Name', 'Label']].copy()
    result['Label'] = result['Label'].astype('Int64')
    result.to_csv(out, index=False)

    total   = len(result)
    barred  = (result['Label'] == 1).sum()
    unbarred= (result['Label'] == 0).sum()
    unknown = result['Label'].isna().sum()

    print(f"\n{inp} -> {out}")
    print(f"Total galaxies: {total}")
    print(f"Barred (1):   {barred}  ({barred/total*100:.1f}%)")
    print(f"Unbarred (0): {unbarred} ({unbarred/total*100:.1f}%)")
    print(f"Unknown:      {unknown}  ({unknown/total*100:.1f}%)")
    return result

print("Processing CSV files...")
print("=" * 60)

train_labels = process_file('train.csv', 'trainlabel.csv')
test_labels  = process_file('test.csv',  'testlabel.csv')

print("\n" + "=" * 60)
print("Done!")
print("\nSample from trainlabel.csv:")
print(train_labels.head(10))
print("\nSample from testlabel.csv:")
print(test_labels.head(10))
